# Recurrent Neural Networks
:label:`sec_rnn`


In :numref:`sec_language-model` we described Markov models and $n$-grams for language modeling, where the conditional probability of token $x_t$ at time step $t$ only depends on the $n-1$ previous tokens.
If we want to incorporate the possible effect of tokens earlier than time step $t-(n-1)$ on $x_t$,
we need to increase $n$.
However, the number of model parameters would also increase exponentially with it, as we need to store $|\mathcal{V}|^n$ numbers for a vocabulary set $\mathcal{V}$.
Hence, rather than modeling $P(x_t \mid x_{t-1}, \ldots, x_{t-n+1})$ it is preferable to use a latent variable model,

$$P(x_t \mid x_{t-1}, \ldots, x_1) \approx P(x_t \mid h_{t-1}),$$

where $h_{t-1}$ is a *hidden state*  that stores the sequence information up to time step $t-1$.
In general,
the hidden state at any time step $t$ could be computed based on both the current input $x_{t}$ and the previous hidden state $h_{t-1}$:

$$h_t = f(x_{t}, h_{t-1}).$$
:eqlabel:`eq_ht_xt`

For a sufficiently powerful function $f$ in :eqref:`eq_ht_xt`, the latent variable model is not an approximation. After all, $h_t$ may simply store all the data it has observed so far.
However, it could potentially make both computation and storage expensive.

Recall that we have discussed hidden layers with hidden units in :numref:`chap_perceptrons`.
It is noteworthy that
hidden layers and hidden states refer to two very different concepts.
Hidden layers are, as explained, layers that are hidden from view on the path from input to output.
Hidden states are technically speaking *inputs* to whatever we do at a given step,
and they can only be computed by looking at data at previous time steps.

*Recurrent neural networks* (RNNs) are neural networks with hidden states. Before introducing the RNN model, we first revisit the MLP model introduced in :numref:`sec_mlp`.



   # 循环神经网络
   :label:`sec_rnn`

   在 :numref:`sec_language-model` 中我们介绍了马尔可夫模型和$n$-元语法在语言建模中的应用，其中时间步$t$的词元$x_t$的条件概率仅依赖于前$n-1$个词元。如果要合并时间步$t-(n-1)$之前的词元对$x_t$的影响，需要增大$n$，但这会导致模型参数量呈指数级增长（需存储$|\mathcal{V}|^n$个参数，$\mathcal{V}$为词表）。因此，我们更倾向于使用隐变量模型：

   $$P(x_t \mid x_{t-1}, \ldots, x_1) \approx P(x_t \mid h_{t-1}),$$

   其中$h_{t-1}$是存储序列到时间步$t-1$信息的*隐藏状态*。通常，任意时间步$t$的隐藏状态可以通过当前输入$x_t$和前一时间步的隐藏状态$h_{t-1}$计算得到：

   $$h_t = f(x_{t}, h_{t-1}).$$
   :eqlabel:`eq_ht_xt`

   当 :eqref:`eq_ht_xt` 中的函数$f$足够强大时，隐变量模型可以视为精确模型（此时$h_t$可存储所有历史观察数据）。但这会导致较高的计算和存储开销。

   请注意：隐藏层（hidden layers）与隐藏状态（hidden states）是两个不同概念。隐藏层指从输入到输出路径中被隐藏的层（详见 :numref:`chap_perceptrons`），而隐藏状态本质上是当前步骤的*输入*，只能通过先前时间步的数据计算得到。

   *循环神经网络*（RNNs）是具有隐藏状态的神经网络。在介绍RNN模型前，我们先回顾 :numref:`sec_mlp` 中的多层感知机模型。

In [1]:
import torch
from d2l import torch as d2l

## Neural Networks without Hidden States

Let's take a look at an MLP with a single hidden layer.
Let the hidden layer's activation function be $\phi$.
Given a minibatch of examples $\mathbf{X} \in \mathbb{R}^{n \times d}$ with batch size $n$ and $d$ inputs, the hidden layer output $\mathbf{H} \in \mathbb{R}^{n \times h}$ is calculated as

$$\mathbf{H} = \phi(\mathbf{X} \mathbf{W}_{\textrm{xh}} + \mathbf{b}_\textrm{h}).$$
:eqlabel:`rnn_h_without_state`

In :eqref:`rnn_h_without_state`, we have the weight parameter $\mathbf{W}_{\textrm{xh}} \in \mathbb{R}^{d \times h}$, the bias parameter $\mathbf{b}_\textrm{h} \in \mathbb{R}^{1 \times h}$, and the number of hidden units $h$, for the hidden layer.
So armed, we apply broadcasting (see :numref:`subsec_broadcasting`) during the summation.
Next, the hidden layer output $\mathbf{H}$ is used as input of the output layer, which is given by

$$\mathbf{O} = \mathbf{H} \mathbf{W}_{\textrm{hq}} + \mathbf{b}_\textrm{q},$$

where $\mathbf{O} \in \mathbb{R}^{n \times q}$ is the output variable, $\mathbf{W}_{\textrm{hq}} \in \mathbb{R}^{h \times q}$ is the weight parameter, and $\mathbf{b}_\textrm{q} \in \mathbb{R}^{1 \times q}$ is the bias parameter of the output layer.  If it is a classification problem, we can use $\mathrm{softmax}(\mathbf{O})$ to compute the probability distribution of the output categories.

This is entirely analogous to the regression problem we solved previously in :numref:`sec_sequence`, hence we omit details.
Suffice it to say that we can pick feature-label pairs at random and learn the parameters of our network via automatic differentiation and stochastic gradient descent.

## Recurrent Neural Networks with Hidden States
:label:`subsec_rnn_w_hidden_states`

Matters are entirely different when we have hidden states. Let's look at the structure in some more detail.

Assume that we have
a minibatch of inputs
$\mathbf{X}_t \in \mathbb{R}^{n \times d}$
at time step $t$.
In other words,
for a minibatch of $n$ sequence examples,
each row of $\mathbf{X}_t$ corresponds to one example at time step $t$ from the sequence.
Next,
denote by $\mathbf{H}_t  \in \mathbb{R}^{n \times h}$ the hidden layer output of time step $t$.
Unlike with MLP, here we save the hidden layer output $\mathbf{H}_{t-1}$ from the previous time step and introduce a new weight parameter $\mathbf{W}_{\textrm{hh}} \in \mathbb{R}^{h \times h}$ to describe how to use the hidden layer output of the previous time step in the current time step. Specifically, the calculation of the hidden layer output of the current time step is determined by the input of the current time step together with the hidden layer output of the previous time step:

$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{\textrm{xh}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hh}}  + \mathbf{b}_\textrm{h}).$$
:eqlabel:`rnn_h_with_state`

Compared with :eqref:`rnn_h_without_state`, :eqref:`rnn_h_with_state` adds one more term $\mathbf{H}_{t-1} \mathbf{W}_{\textrm{hh}}$ and thus
instantiates :eqref:`eq_ht_xt`.
From the relationship between hidden layer outputs $\mathbf{H}_t$ and $\mathbf{H}_{t-1}$ of adjacent time steps,
we know that these variables captured and retained the sequence's historical information up to their current time step, just like the state or memory of the neural network's current time step. Therefore, such a hidden layer output is called a *hidden state*.
Since the hidden state uses the same definition of the previous time step in the current time step, the computation of :eqref:`rnn_h_with_state` is *recurrent*. Hence, as we said, neural networks with hidden states
based on recurrent computation are named
*recurrent neural networks*.
Layers that perform
the computation of :eqref:`rnn_h_with_state`
in RNNs
are called *recurrent layers*.


There are many different ways for constructing RNNs.
Those with a hidden state defined by :eqref:`rnn_h_with_state` are very common.
For time step $t$,
the output of the output layer is similar to the computation in the MLP:

$$\mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{\textrm{hq}} + \mathbf{b}_\textrm{q}.$$

Parameters of the RNN
include the weights $\mathbf{W}_{\textrm{xh}} \in \mathbb{R}^{d \times h}, \mathbf{W}_{\textrm{hh}} \in \mathbb{R}^{h \times h}$,
and the bias $\mathbf{b}_\textrm{h} \in \mathbb{R}^{1 \times h}$
of the hidden layer,
together with the weights $\mathbf{W}_{\textrm{hq}} \in \mathbb{R}^{h \times q}$
and the bias $\mathbf{b}_\textrm{q} \in \mathbb{R}^{1 \times q}$
of the output layer.
It is worth mentioning that
even at different time steps,
RNNs always use these model parameters.
Therefore, the parametrization cost of an RNN
does not grow as the number of time steps increases.

:numref:`fig_rnn` illustrates the computational logic of an RNN at three adjacent time steps.
At any time step $t$,
the computation of the hidden state can be treated as:
(i) concatenating the input $\mathbf{X}_t$ at the current time step $t$ and the hidden state $\mathbf{H}_{t-1}$ at the previous time step $t-1$;
(ii) feeding the concatenation result into a fully connected layer with the activation function $\phi$.
The output of such a fully connected layer is the hidden state $\mathbf{H}_t$ of the current time step $t$.
In this case,
the model parameters are the concatenation of $\mathbf{W}_{\textrm{xh}}$ and $\mathbf{W}_{\textrm{hh}}$, and a bias of $\mathbf{b}_\textrm{h}$, all from :eqref:`rnn_h_with_state`.
The hidden state of the current time step $t$, $\mathbf{H}_t$, will participate in computing the hidden state $\mathbf{H}_{t+1}$ of the next time step $t+1$.
What is more, $\mathbf{H}_t$ will also be
fed into the fully connected output layer
to compute the output
$\mathbf{O}_t$ of the current time step $t$.

![An RNN with a hidden state.](../img/rnn.svg)
:label:`fig_rnn`

We just mentioned that the calculation of $\mathbf{X}_t \mathbf{W}_{\textrm{xh}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hh}}$ for the hidden state is equivalent to
matrix multiplication of the
concatenation of $\mathbf{X}_t$ and $\mathbf{H}_{t-1}$
and the
concatenation of $\mathbf{W}_{\textrm{xh}}$ and $\mathbf{W}_{\textrm{hh}}$.
Though this can be proven mathematically,
in the following we just use a simple code snippet as a demonstration.
To begin with,
we define matrices `X`, `W_xh`, `H`, and `W_hh`, whose shapes are (3, 1), (1, 4), (3, 4), and (4, 4), respectively.
Multiplying `X` by `W_xh`, and `H` by `W_hh`, and then adding these two products,
we obtain a matrix of shape (3, 4).



## 无隐藏状态的神经网络

让我们从具有单个隐藏层的MLP开始。假设隐藏层的激活函数为$\phi$。给定小批量样本$\mathbf{X} \in \mathbb{R}^{n \times d}$（批量大小$n$，输入维度$d$），隐藏层输出$\mathbf{H} \in \mathbb{R}^{n \times h}$计算为：

$$\mathbf{H} = \phi(\mathbf{X} \mathbf{W}_{\textrm{xh}} + \mathbf{b}_\textrm{h}).$$
:eqlabel:`rnn_h_without_state`

在 :eqref:`rnn_h_without_state`中：
- $\mathbf{W}_{\textrm{xh}} \in \mathbb{R}^{d \times h}$为隐藏层权重
- $\mathbf{b}_\textrm{h} \in \mathbb{R}^{1 \times h}$为偏置参数
- $h$为隐藏单元数

接着将隐藏层输出$\mathbf{H}$作为输出层的输入：
$$\mathbf{O} = \mathbf{H} \mathbf{W}_{\textrm{hq}} + \mathbf{b}_\textrm{q},$$
其中$\mathbf{O} \in \mathbb{R}^{n \times q}$为输出变量。对于分类问题，可用$\mathrm{softmax}(\mathbf{O})$计算输出类别的概率分布。

这与 :numref:`sec_sequence`中的回归问题解决方法完全类似，因此我们省略细节。通过随机选取特征-标签对，并借助自动微分和随机梯度下降即可学习网络参数。

## 具有隐藏状态的循环神经网络
:label:`subsec_rnn_w_hidden_states`

当引入隐藏状态时，情况将发生本质变化。假设在时间步$t$有小批量输入$\mathbf{X}_t \in \mathbb{R}^{n \times d}$，$\mathbf{H}_t  \in \mathbb{R}^{n \times h}$表示时间步$t$的隐藏层输出。与MLP不同，这里我们保存前一时间步的隐藏层输出$\mathbf{H}_{t-1}$，并引入新的权重参数$\mathbf{W}_{\textrm{hh}} \in \mathbb{R}^{h \times h}$来描述如何利用前一时间步的隐藏状态：

$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{\textrm{xh}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hh}}  + \mathbf{b}_\textrm{h}).$$
:eqlabel:`rnn_h_with_state`

与 :eqref:`rnn_h_without_state`相比，:eqref:`rnn_h_with_state`增加了$\mathbf{H}_{t-1} \mathbf{W}_{\textrm{hh}}$项，实现了 :eqref:`eq_ht_xt`。通过相邻时间步隐藏状态$\mathbf{H}_t$与$\mathbf{H}_{t-1}$的关系，这些变量捕获并保留了序列的历史信息，类似于神经网络的"记忆"。因此这种隐藏层输出被称为*隐藏状态*。由于隐藏状态的计算具有时间递归特性，基于这种循环计算的神经网络称为*循环神经网络*（RNNs），执行 :eqref:`rnn_h_with_state`计算的层称为*循环层*。

时间步$t$的输出层计算与MLP类似：
$$\mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{\textrm{hq}} + \mathbf{b}_\textrm{q}.$$

RNN参数包括：
- 隐藏层权重 $\mathbf{W}_{\textrm{xh}} \in \mathbb{R}^{d \times h}$, $\mathbf{W}_{\textrm{hh}} \in \mathbb{R}^{h \times h}$
- 隐藏层偏置 $\mathbf{b}_\textrm{h} \in \mathbb{R}^{1 \times h}$ 
- 输出层权重 $\mathbf{W}_{\textrm{hq}} \in \mathbb{R}^{h \times q}$
- 输出层偏置 $\mathbf{b}_\textrm{q} \in \mathbb{R}^{1 \times q}$

值得注意的是，不同时间步的RNN使用相同参数，因此参数数量不随时间步增加而增长。

:numref:`fig_rnn`展示了相邻三个时间步的RNN计算逻辑。任意时间步$t$的隐藏状态计算可分解为：
1. 将当前输入$\mathbf{X}_t$与前一时间步隐藏状态$\mathbf{H}_{t-1}$拼接
2. 输入到具有激活函数$\phi$的全连接层

当前隐藏状态$\mathbf{H}_t$将参与下一时间步的计算，并输入到输出层产生$\mathbf{O}_t$。

![具有隐藏状态的循环神经网络结构图](../img/rnn.svg)
:label:`fig_rnn`

需要说明的是，$\mathbf{X}_t \mathbf{W}_{\textrm{xh}} + \mathbf{H}_{t-1} \mathbf{W}_{\textrm{hh}}$的计算等效于拼接矩阵的矩阵乘法。以下代码片段演示该计算过程（保留原始代码不变）：

```python
# 定义矩阵参数（保持原始代码注释）
X = np.array([[1], [2], [3]])
W_xh = np.random.randn(1, 4)
H = np.random.randn(3, 4)
W_hh = np.random.randn(4, 4)

# 执行矩阵运算（保持代码结构不变）
output = np.dot(X, W_xh) + np.dot(H, W_hh)
```

In [2]:
X, W_xh = torch.randn(3, 1), torch.randn(1, 4)
H, W_hh = torch.randn(3, 4), torch.randn(4, 4)
torch.matmul(X, W_xh) + torch.matmul(H, W_hh)

tensor([[ 2.4117, -2.8091,  6.0713,  0.9842],
        [ 2.8806,  0.9823,  0.4846, -0.4477],
        [ 0.8806, -1.7184, -1.5466, -1.7534]])

Now we concatenate the matrices `X` and `H`
along columns (axis 1),
and the matrices
`W_xh` and `W_hh` along rows (axis 0).
These two concatenations
result in
matrices of shape (3, 5)
and of shape (5, 4), respectively.
Multiplying these two concatenated matrices,
we obtain the same output matrix of shape (3, 4)
as above.



现在我们沿列（轴1）拼接矩阵`X`和`H`，沿行（轴0）拼接矩阵`W_xh`和`W_hh`。这两个拼接操作将分别生成形状为(3, 5)和(5, 4)的矩阵。相乘这两个拼接后的矩阵，我们得到与前述方法相同的形状为(3, 4)的输出矩阵。

In [3]:
torch.matmul(torch.cat((X, H), 1), torch.cat((W_xh, W_hh), 0))

tensor([[ 2.4117, -2.8091,  6.0713,  0.9842],
        [ 2.8806,  0.9823,  0.4846, -0.4477],
        [ 0.8806, -1.7184, -1.5466, -1.7534]])

## RNN-Based Character-Level Language Models

Recall that for language modeling in :numref:`sec_language-model`,
we aim to predict the next token based on
the current and past tokens;
thus we shift the original sequence by one token
as the targets (labels).
:citet:`Bengio.Ducharme.Vincent.ea.2003` first proposed
to use a neural network for language modeling.
In the following we illustrate how RNNs can be used to build a language model.
Let the minibatch size be one, and the sequence of the text be "machine".
To simplify training in subsequent sections,
we tokenize text into characters rather than words
and consider a *character-level language model*.
:numref:`fig_rnn_train` demonstrates how to predict the next character based on the current and previous characters via an RNN for character-level language modeling.

![A character-level language model based on the RNN. The input and target sequences are "machin" and "achine", respectively.](../img/rnn-train.svg)
:label:`fig_rnn_train`

During the training process,
we run a softmax operation on the output from the output layer for each time step, and then use the cross-entropy loss to compute the error between the model output and the target.
Because of the recurrent computation of the hidden state in the hidden layer, the output, $\mathbf{O}_3$,  of time step 3 in :numref:`fig_rnn_train` is determined by the text sequence "m", "a", and "c". Since the next character of the sequence in the training data is "h", the loss of time step 3 will depend on the probability distribution of the next character generated based on the feature sequence "m", "a", "c" and the target "h" of this time step.

In practice, each token is represented by a $d$-dimensional vector, and we use a batch size $n>1$. Therefore, the input $\mathbf X_t$ at time step $t$ will be an $n\times d$ matrix, which is identical to what we discussed in :numref:`subsec_rnn_w_hidden_states`.

In the following sections, we will implement RNNs
for character-level language models.


## Summary

A neural network that uses recurrent computation for hidden states is called a recurrent neural network (RNN).
The hidden state of an RNN can capture historical information of the sequence up to the current time step. With recurrent computation, the number of RNN model parameters does not grow as the number of time steps increases. As for applications, an RNN can be used to create character-level language models.


## Exercises

1. If we use an RNN to predict the next character in a text sequence, what is the required dimension for any output?
1. Why can RNNs express the conditional probability of a token at some time step based on all the previous tokens in the text sequence?
1. What happens to the gradient if you backpropagate through a long sequence?
1. What are some of the problems associated with the language model described in this section?



## 基于RNN的字符级语言模型

回顾 :numref:`sec_language-model`中的语言建模任务，我们的目标是根据当前及历史词元预测下一个词元。:citet:`Bengio.Ducharme.Vincent.ea.2003`首次提出使用神经网络进行语言建模。以下我们将阐述如何用RNN构建语言模型。设小批量大小为1，文本序列为"machine"。为简化后续章节的训练，我们将文本按字符而非词元分割，构建*字符级语言模型*。:numref:`fig_rnn_train`展示了如何通过RNN基于当前及历史字符预测下一个字符。

![基于RNN的字符级语言模型。输入序列和目标序列分别为"machin"和"achine"](../img/rnn-train.svg)
:label:`fig_rnn_train`

训练过程中，我们对每个时间步的输出层结果执行softmax运算，并使用交叉熵损失计算模型输出与目标的误差。由于隐藏层的循环计算特性，:numref:`fig_rnn_train`中时间步3的输出$\mathbf{O}_3$由序列"m"、"a"、"c"决定。因训练数据中该序列的下个字符为"h"，时间步3的损失将取决于基于特征序列"m"、"a"、"c"生成的下个字符概率分布与目标"h"的差异。

实际应用中，每个词元用$d$维向量表示，批量大小为$n>1$。因此时间步$t$的输入$\mathbf X_t$为$n\times d$矩阵，与:numref:`subsec_rnn_w_hidden_states`讨论的情况一致。

后续章节我们将实现字符级语言模型的RNN。

## 小结

使用循环计算隐藏状态的神经网络称为循环神经网络(RNN)。RNN的隐藏状态可捕获序列到当前时间步的历史信息。通过循环计算，RNN模型参数数量不随时间步增加而增长。应用方面，RNN可用于构建字符级语言模型。

## 练习

1. 若使用RNN预测文本序列的下个字符，输出需要多少维？
2. 为何RNN能表示某时间步词元基于文本序列中所有前驱词元的条件概率？
3. 若通过长序列反向传播，梯度会发生什么变化？
4. 本节描述的语言模型存在哪些潜在问题？

[Discussions](https://discuss.d2l.ai/t/1050)
